# **Inference**

## **1. Importing stuff**

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import librosa
from typing import Optional, Dict, List, Any
from pathlib import Path
import math
import soundfile as sf
from tqdm import tqdm
from IPython.display import display, Audio

from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.models.factorization import LocalEncoder, GlobalEncoder
from SCAPES.models.factorization import load_local_encoder, load_global_encoder
from SCAPES.models.flow import FlowModel, load_flow_model
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper

from SCAPES.inference.FlowInference import FlowInference

# Spherical interpolation 
def slerp(v0, v1, alpha, eps=1e-7):
    """
    Spherical linear interpolation between two normalized vectors.
    v0, v1: (..., D)
    alpha: scalar in [0,1]
    """
    v0 = v0 / v0.norm(p=2)
    v1 = v1 / v1.norm(p=2)

    dot = torch.clamp(torch.dot(v0, v1), -1.0 + eps, 1.0 - eps)
    theta = torch.acos(dot)

    if theta < eps:
        # vectors are almost identical; fall back to lerp
        return (1 - alpha) * v0 + alpha * v1

    sin_theta = torch.sin(theta)
    w0 = torch.sin((1 - alpha) * theta) / sin_theta
    w1 = torch.sin(alpha * theta) / sin_theta

    return w0 * v0 + w1 * v1

def load_and_encode(engine, audio_path, max_samples=48000*20):
    audio_tensor = engine.load_audio_to_tensor(audio_path)[:, :, :max_samples]
    atoms = engine.encode_audio_to_atoms(audio_tensor)
    contexts = engine.compute_context_track(atoms)
    return atoms, contexts

# Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# Initialize processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

## **2. Load models**

Make sure you point to the model checkpoints here

In [ ]:
# Flow model path
model_checkpoints = Path("path/to/your/model")  # <-- UPDATE THIS PATH

# Create paths for models and configs
flow_ckpt_path   = model_checkpoints / "checkpoints" / "best_flow_model.pt"
flow_config_path = model_checkpoints / "checkpoints" / "flow_model_config.json"
local_ckpt_path   = model_checkpoints / "checkpoints" / "best_local_encoder.pt"
local_config_path = model_checkpoints / "checkpoints" / "local_encoder_config.json"

# Path for the global encoder
global_ckpt_path   = Path("../SCAPES/weights/factorization/GlobalEncoder_universal.pt")
global_config_path = Path("../SCAPES/weights/factorization/GlobalEncoder_universal.json")

# Load checkpoints and configs
local_encoder = load_local_encoder(
    checkpoint_path=local_ckpt_path, 
    json_path=local_config_path, 
    device=device
)
flow_model = load_flow_model(
    checkpoint_path=flow_ckpt_path, 
    json_path=flow_config_path, 
    device=device
)
global_encoder = load_global_encoder(
    checkpoint_path=global_ckpt_path, 
    json_path=global_config_path, 
    device=device
)
clap_model = CLAPWrapper(version="2023", use_cuda=True)

# Load the inference engine
engine = FlowInference(
    model=flow_model,
    local_encoder=local_encoder,
    processor=processor_48k,
    context_model=clap_model,  # Passing clap or proxy
    segment_length=segment_length,
    context_length=context_length,
    atoms_frames=21,
    atoms_overlap_frames=3,
    device=device,
    verbose=True
)

## **3. Audio generation**

In [ ]:
def run_interpolation_pipeline(
    engine,
    audio_path_1,
    audio_path_2,
    timeline_size=200,
    stay_time=30,
    mode="slerp",  # "linear" or "slerp"
    play=True,
    decode_method="ola"
):
    # Load both audios
    atoms_1, contexts_1 = load_and_encode(engine, audio_path_1)
    atoms_2, contexts_2 = load_and_encode(engine, audio_path_2)

    c0 = contexts_1[0]
    c1 = contexts_2[0]

    atoms = [None] * timeline_size
    contexts = []

    for t in range(timeline_size):
        if t < stay_time:
            contexts.append(c0)

        elif t < timeline_size - stay_time:
            alpha = (t - stay_time) / (timeline_size - 2 * stay_time)

            if mode == "linear":
                blended = (1 - alpha) * c0 + alpha * c1
                blended = blended / blended.norm(p=2)

            elif mode == "slerp":
                blended = slerp(c0, c1, alpha)

            else:
                raise ValueError("mode must be 'linear' or 'slerp'")

            contexts.append(blended)

        else:
            contexts.append(c1)

    # Build timeline
    timeline = engine.build_base_timeline(
        atoms_129D=atoms,
        context_embeddings=contexts,
        default_TF=False,
        default_AF=0.0
    )

    # Generate
    completed_timeline = engine.generate(timeline, NFE=32)

    # Decode
    final_wav = engine.decode_timeline(completed_timeline, output_path=None, method=decode_method)

    if play:
        print(f"Interpolation ({mode}): {Path(audio_path_1).stem} -> {Path(audio_path_2).stem}")
        display(Audio(final_wav, rate=engine.sr))

    return final_wav

def run_resynthesis_pipeline(
    engine,
    audio_path,
    timeline_size=200,
    play=True,
    decode_method="ola"
):
    atoms_src, contexts_src = load_and_encode(engine, audio_path)

    c0 = contexts_src[0]

    atoms = [None] * timeline_size
    contexts = [c0 for _ in range(timeline_size)]

    timeline = engine.build_base_timeline(
        atoms_129D=atoms,
        context_embeddings=contexts,
        default_TF=False,
        default_AF=0.0
    )

    completed_timeline = engine.generate(timeline, NFE=32)
    final_wav = engine.decode_timeline(completed_timeline, output_path=None, method=decode_method)

    if play:
        # pick filename without extension for display
        filename = Path(audio_path).stem
        print("Resynthesis: ", filename)
        display(Audio(final_wav, rate=engine.sr))

    return final_wav

In [ ]:
# paths = {
#     "keyboard": "../../datasets/hand_curated/raw/keyboard.wav",
#     "fire": "../../datasets/hand_curated/raw/fire.wav",
#     "waterfall": "../../datasets/hand_curated/raw/waterfall.wav",
#     "river": "../../datasets/hand_curated/raw/river.wav",
    # "shards": "../../datasets/hand_curated/raw/shards.wav",
    # "bubbles": "../../datasets/hand_curated/raw/bubbles.wav",
    # "wind": "../../datasets/hand_curated/raw/wind.wav",
    # "rain": "../../datasets/hand_curated/raw/rain.wav"
# }

# for name, path in paths.items():
#     # print(f"Running resynthesis pipeline for {name}...")
#     run_resynthesis_pipeline(engine, path, timeline_size=30, play=True, decode_method="latent_stitch")

# # make interpolation for all pairs
# for name1, path1 in paths.items():
#     for name2, path2 in paths.items():
#         if name1 < name2:  # Avoid duplicates and self-pairs
#             # print(f"Running interpolation pipeline for {name1} -> {name2}...")
#             run_interpolation_pipeline(engine, path1, path2, timeline_size=166, stay_time=33, mode="slerp", play=True)